In [1]:
import os, gc, glob, hashlib
import numpy as np
import pandas as pd
import nibabel as nib
from scipy.ndimage import zoom
from sklearn.model_selection import train_test_split
from tqdm import tqdm

DATASET_ROOT = os.environ.get("DATASET_ROOT", "/home/jovyan/shared/donut/Alzheimer/OAS1")
SEEDS = [1334, 0, 1, 42, 2024, 3407]
CHECK_ID = "0454"

SIZE_2D = 224
SHAPE_3D = (128, 128, 96)
MODES = ["2D", "2.5D", "3D"]
EXPECTED = {"2D": (1, SIZE_2D, SIZE_2D), "2.5D": (3, SIZE_2D, SIZE_2D), "3D": (1,) + SHAPE_3D}


In [2]:
df = pd.read_excel(f"{DATASET_ROOT}/_oasis_cross-sectional.xlsx")
df = df[df["CDR"].notna()]
df["label"] = (df["CDR"] > 0).astype(int)

def find_file(subject_id):
    pattern = f"{DATASET_ROOT}/{subject_id}/PROCESSED/MPRAGE/T88_111/*_masked_gfc.hdr"
    files = glob.glob(pattern)
    return files[0] if files else None

df["path"] = df["ID"].apply(find_file)
df = df[df["path"].notna()]
df["subject_id"] = df["ID"].apply(lambda x: x.split("_")[1])
print("df หลังกรอง:", df.shape, "| subject ไม่ซ้ำ:", df["subject_id"].nunique(), "|", CHECK_ID, "อยู่ใน df:", CHECK_ID in set(df["subject_id"]))


df หลังกรอง: (234, 15) | subject ไม่ซ้ำ: 234 | 0454 อยู่ใน df: True


In [3]:
def split_ids(df, seed):
    # ลอจิกเดียวกับโน้ตบุ๊กทั้ง 3 โมเดล
    sl = df.groupby("subject_id")["label"].first()
    ids, vals = list(sl.index), list(sl.values)
    tr, temp = train_test_split(ids, test_size=0.3, stratify=vals, random_state=seed)
    lab = dict(zip(ids, vals))
    va, te = train_test_split(temp, test_size=0.5, stratify=[lab[s] for s in temp], random_state=seed)
    a, b, c = set(tr), set(va), set(te)
    assert not (a & b) and not (a & c) and not (b & c)
    return a, b, c

def load_and_prep(path):
    # crop สมอง + normalize เหมือนโน้ตบุ๊กจริง (ยังไม่ resize)
    v = nib.load(path).get_fdata()
    if v.ndim == 4:
        v = v[:, :, :, 0]
    nz = np.argwhere(v > 0)
    lo, hi = nz.min(axis=0), nz.max(axis=0)
    v = v[lo[0]:hi[0]+1, lo[1]:hi[1]+1, lo[2]:hi[2]+1]
    p_lo, p_hi = np.percentile(v, [1, 99])
    v = np.clip(v, p_lo, p_hi)
    return (v - p_lo) / (p_hi - p_lo + 1e-8)

def resize2d(s):
    return zoom(s, (SIZE_2D / s.shape[0], SIZE_2D / s.shape[1]), order=1)

def to_target(v, mode):
    if mode == "3D":
        return zoom(v, [t / c for t, c in zip(SHAPE_3D, v.shape)], order=1)[None]       # (1,128,128,96)
    if mode == "2D":
        return resize2d(v[:, :, v.shape[2] // 2])[None]                                  # (1,224,224)
    if mode == "2.5D":
        cx, cy, cz = [s // 2 for s in v.shape]
        return np.stack([resize2d(v[cx, :, :]), resize2d(v[:, cy, :]), resize2d(v[:, :, cz])])  # (3,224,224)
    raise ValueError(mode)

def checksum(a):
    return hashlib.sha1(np.ascontiguousarray(a).tobytes()).hexdigest()[:10]

def build_cache(mode):
    # อ่านและประมวลผลแต่ละไฟล์ครั้งเดียว (ผลไม่ขึ้นกับ seed)
    cache = {}
    for sid, path in tqdm(zip(df["subject_id"], df["path"]), total=len(df), desc=f"โหลด {mode}"):
        cache[sid] = to_target(load_and_prep(path), mode).astype(np.float32)
    return cache

def assemble(cache, ids):
    ids = [s for s in df["subject_id"] if s in ids]      # ลำดับเดียวกับ df
    return np.stack([cache[s] for s in ids]), ids


In [4]:
# สัดส่วนปกติ / อัลไซเมอร์ ในแต่ละชุด ของแต่ละ seed (นับจากรายชื่อ ไม่ต้องโหลดภาพ)
lab = df.groupby("subject_id")["label"].first()
tot_ad, tot_nm = int((lab == 1).sum()), int((lab == 0).sum())
print(f"ทั้งหมด: ปกติ={tot_nm} อัลไซเมอร์={tot_ad} รวม={tot_nm + tot_ad} | % อัลไซเมอร์={100 * tot_ad / (tot_ad + tot_nm):.1f}")

rows = []
for sd in SEEDS:
    for name, ids in zip(("train", "val", "test"), split_ids(df, sd)):
        n_ad = int(lab.loc[list(ids)].sum())
        n_nm = len(ids) - n_ad
        rows.append({"seed": sd, "ชุด": name, "ปกติ": n_nm, "อัลไซเมอร์": n_ad, "รวม": len(ids),
                     "% อัลไซเมอร์": round(100 * n_ad / len(ids), 1),
                     "% ของปกติทั้งหมด": round(100 * n_nm / tot_nm, 1),
                     "% ของอัลไซเมอร์ทั้งหมด": round(100 * n_ad / tot_ad, 1)})
ratio_tab = pd.DataFrame(rows)
print(ratio_tab.to_string(index=False))
print()
print("ควรเห็น: % อัลไซเมอร์ ใกล้เคียงกันทั้ง 3 ชุด และ % ของกลุ่มทั้งหมด ใกล้ 70/15/15 (ต่างกันไม่เกิน 1 คน)")


ทั้งหมด: ปกติ=134 อัลไซเมอร์=100 รวม=234 | % อัลไซเมอร์=42.7
 seed   ชุด  ปกติ  อัลไซเมอร์  รวม  % อัลไซเมอร์  % ของปกติทั้งหมด  % ของอัลไซเมอร์ทั้งหมด
 1334 train    93          70  163          42.9              69.4                    70.0
 1334   val    20          15   35          42.9              14.9                    15.0
 1334  test    21          15   36          41.7              15.7                    15.0
    0 train    93          70  163          42.9              69.4                    70.0
    0   val    20          15   35          42.9              14.9                    15.0
    0  test    21          15   36          41.7              15.7                    15.0
    1 train    93          70  163          42.9              69.4                    70.0
    1   val    20          15   35          42.9              14.9                    15.0
    1  test    21          15   36          41.7              15.7                    15.0
   42 train    93          70

In [5]:
records = []
split_ids_by_seed = {sd: split_ids(df, sd) for sd in SEEDS}

for mode in MODES:
    print("#" * 78)
    print(f"โมเดล {mode}  target shape = {EXPECTED[mode]}")
    cache = build_cache(mode)

    for sd in SEEDS:
        a, b, c = split_ids_by_seed[sd]
        x_tr, s_tr = assemble(cache, a)
        x_va, s_va = assemble(cache, b)
        x_te, s_te = assemble(cache, c)

        for nm, x in [("train", x_tr), ("val", x_va), ("test", x_te)]:
            assert x.shape[1:] == EXPECTED[mode], f"{mode} {nm} shape ผิด: {x.shape}"

        loaded = {"train": (s_tr, x_tr), "val": (s_va, x_va), "test": (s_te, x_te)}
        hits = [(nm, sids.index(CHECK_ID), checksum(x[sids.index(CHECK_ID)]))
                for nm, (sids, x) in loaded.items() if CHECK_ID in sids]
        where, idx, ck = ("ไม่พบ", None, None) if not hits else hits[0]

        split_hash = hashlib.sha1("|".join(",".join(sorted(s)) for s in (a, b, c)).encode()).hexdigest()[:10]
        print(f"  {mode:5s} seed={sd:<5} train={len(s_tr)} val={len(s_va)} test={len(s_te)} x_train{tuple(x_tr.shape)} | {CHECK_ID} -> {where}"
              + (f" (ตำแหน่ง {idx}, checksum {ck})" if idx is not None else ""))
        records.append({"mode": mode, "seed": sd, "where": where, "index": idx,
                        "image_checksum": ck, "split_hash": split_hash})
        del x_tr, x_va, x_te, loaded

    del cache
    gc.collect()


##############################################################################
โมเดล 2D  target shape = (1, 224, 224)


โหลด 2D: 100%|██████████| 234/234 [00:12<00:00, 18.06it/s]


  2D    seed=1334  train=163 val=35 test=36 x_train(163, 1, 224, 224) | 0454 -> val (ตำแหน่ง 34, checksum d8689fbdcd)
  2D    seed=0     train=163 val=35 test=36 x_train(163, 1, 224, 224) | 0454 -> train (ตำแหน่ง 161, checksum d8689fbdcd)
  2D    seed=1     train=163 val=35 test=36 x_train(163, 1, 224, 224) | 0454 -> train (ตำแหน่ง 160, checksum d8689fbdcd)
  2D    seed=42    train=163 val=35 test=36 x_train(163, 1, 224, 224) | 0454 -> train (ตำแหน่ง 160, checksum d8689fbdcd)
  2D    seed=2024  train=163 val=35 test=36 x_train(163, 1, 224, 224) | 0454 -> train (ตำแหน่ง 161, checksum d8689fbdcd)
  2D    seed=3407  train=163 val=35 test=36 x_train(163, 1, 224, 224) | 0454 -> test (ตำแหน่ง 35, checksum d8689fbdcd)
##############################################################################
โมเดล 2.5D  target shape = (3, 224, 224)


โหลด 2.5D: 100%|██████████| 234/234 [00:12<00:00, 18.78it/s]


  2.5D  seed=1334  train=163 val=35 test=36 x_train(163, 3, 224, 224) | 0454 -> val (ตำแหน่ง 34, checksum 08ef4211ff)
  2.5D  seed=0     train=163 val=35 test=36 x_train(163, 3, 224, 224) | 0454 -> train (ตำแหน่ง 161, checksum 08ef4211ff)
  2.5D  seed=1     train=163 val=35 test=36 x_train(163, 3, 224, 224) | 0454 -> train (ตำแหน่ง 160, checksum 08ef4211ff)
  2.5D  seed=42    train=163 val=35 test=36 x_train(163, 3, 224, 224) | 0454 -> train (ตำแหน่ง 160, checksum 08ef4211ff)
  2.5D  seed=2024  train=163 val=35 test=36 x_train(163, 3, 224, 224) | 0454 -> train (ตำแหน่ง 161, checksum 08ef4211ff)
  2.5D  seed=3407  train=163 val=35 test=36 x_train(163, 3, 224, 224) | 0454 -> test (ตำแหน่ง 35, checksum 08ef4211ff)
##############################################################################
โมเดล 3D  target shape = (1, 128, 128, 96)


โหลด 3D: 100%|██████████| 234/234 [00:19<00:00, 12.07it/s]


  3D    seed=1334  train=163 val=35 test=36 x_train(163, 1, 128, 128, 96) | 0454 -> val (ตำแหน่ง 34, checksum 06f99be104)
  3D    seed=0     train=163 val=35 test=36 x_train(163, 1, 128, 128, 96) | 0454 -> train (ตำแหน่ง 161, checksum 06f99be104)
  3D    seed=1     train=163 val=35 test=36 x_train(163, 1, 128, 128, 96) | 0454 -> train (ตำแหน่ง 160, checksum 06f99be104)
  3D    seed=42    train=163 val=35 test=36 x_train(163, 1, 128, 128, 96) | 0454 -> train (ตำแหน่ง 160, checksum 06f99be104)
  3D    seed=2024  train=163 val=35 test=36 x_train(163, 1, 128, 128, 96) | 0454 -> train (ตำแหน่ง 161, checksum 06f99be104)
  3D    seed=3407  train=163 val=35 test=36 x_train(163, 1, 128, 128, 96) | 0454 -> test (ตำแหน่ง 35, checksum 06f99be104)


In [6]:
summary = pd.DataFrame(records)
print(summary.to_string(index=False))
print()

print("1) ภายในโมเดลเดียว: ภาพของ", CHECK_ID, "เหมือนเดิมทุก seed ไหม (ควรเป็น True)")
for m, g in summary.groupby("mode", sort=False):
    ck = g["image_checksum"].dropna()
    print(f"   {m:5s}: {'ไม่มีข้อมูล (ไม่อยู่ใน df)' if ck.empty else ck.nunique() == 1}"
          f" | ชุดที่อยู่ตาม seed: {dict(zip(g['seed'], g['where']))}")

print()
print("2) seed เดียวกัน ทั้ง 3 โมเดล: รายชื่อ+ชุดเหมือนกันไหม (ควรเป็น True ทุก seed)")
for sd, g in summary.groupby("seed", sort=False):
    print(f"   seed={sd:<5}: split_hash เหมือนกัน={g['split_hash'].nunique() == 1} | {CHECK_ID} -> {g['where'].unique().tolist()}")

print()
print("3) ถ้าโมเดลแต่ละตัวใช้ seed ไม่เหมือนกัน (แก้ MODEL_SEEDS ได้)")
MODEL_SEEDS = {"2D": 1334, "2.5D": 42, "3D": 2024}
pick = {m: split_ids_by_seed[s] for m, s in MODEL_SEEDS.items()}
def where_of(sp, sid):
    return next((n for n, s in zip(("train", "val", "test"), sp) if sid in s), "ไม่พบ")
for m, sp in pick.items():
    print(f"   {m:5s} seed={MODEL_SEEDS[m]:<5} {CHECK_ID} -> {where_of(sp, CHECK_ID)}")
conf = [s for s in df["subject_id"] if len({where_of(sp, s) for sp in pick.values()}) > 1]
print(f"   คนที่อยู่ชุดไม่ตรงกันข้ามโมเดล: {len(conf)} / {len(df)} คน")


mode  seed where  index image_checksum split_hash
  2D  1334   val     34     d8689fbdcd 67c0d18d36
  2D     0 train    161     d8689fbdcd 0db4337943
  2D     1 train    160     d8689fbdcd c2cd387558
  2D    42 train    160     d8689fbdcd e83b75a66e
  2D  2024 train    161     d8689fbdcd f8281add23
  2D  3407  test     35     d8689fbdcd ab725e95f3
2.5D  1334   val     34     08ef4211ff 67c0d18d36
2.5D     0 train    161     08ef4211ff 0db4337943
2.5D     1 train    160     08ef4211ff c2cd387558
2.5D    42 train    160     08ef4211ff e83b75a66e
2.5D  2024 train    161     08ef4211ff f8281add23
2.5D  3407  test     35     08ef4211ff ab725e95f3
  3D  1334   val     34     06f99be104 67c0d18d36
  3D     0 train    161     06f99be104 0db4337943
  3D     1 train    160     06f99be104 c2cd387558
  3D    42 train    160     06f99be104 e83b75a66e
  3D  2024 train    161     06f99be104 f8281add23
  3D  3407  test     35     06f99be104 ab725e95f3

1) ภายในโมเดลเดียว: ภาพของ 0454 เหมือนเดิมทุก see